In [1]:
import re
import json
from pathlib import Path

def load_vocabulary(vocab_path: str = "../data/vocabulary.json") -> dict:
    """
    Load dynamic vocabulary built by build_vocabulary.py.
    Falls back to a minimal hardcoded seed if the file is missing,
    so the script never fails silently on a fresh environment.
    """
    path = Path(vocab_path)

    if not path.exists():
        return {
            "filled_pauses"    : ["uh", "um", "hmm", "eh", "anu", "ee", "eee", "hm", "mm"],
            "repair_bridges"   : ["maksudnya", "sebenarnya", "sebenernya",
                                  "pokoknya", "intinya", "soalnya", "bukan", "maaf", "wait"],
            "discourse_markers": ["yaudah", "dong", "gitu", "nih", "aja", "deh", "sih", "ya"],
            "partial_word_re"  : r"^[a-zA-Z]+-$",
        }

    with open(path, encoding="utf-8") as f:
        vocab = json.load(f)
    return vocab


# Load at module level — used by all labelers below
_VOCAB            = load_vocabulary()
FILLED_PAUSES     = set(_VOCAB["filled_pauses"])
REPAIR_BRIDGES    = _VOCAB["repair_bridges"]      # already sorted longest-first
DISCOURSE_MARKERS = _VOCAB["discourse_markers"]   # already sorted longest-first
PARTIAL_WORD_RE   = re.compile(_VOCAB["partial_word_re"])

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# TEXT NORMALIZATION
# ─────────────────────────────────────────────────────────────────────────────
 
def normalize_and_tokenize(text: str) -> list[str]:
    """
    Lowercase, strip leading/trailing whitespace, remove punctuation that
    would split tokens incorrectly, then split on whitespace.
 
    We preserve hyphens INSIDE tokens (e.g. "rica-rica", "nasi goreng pete-nya")
    and at the END of tokens (partial words like "na-").
    We strip commas, periods, question marks, exclamation marks.
    """
    text = text.lower().strip()
    # Remove sentence-final and clause-final punctuation
    text = re.sub(r"[,\.\!\?]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    return text.split()

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# MATCHING HELPERS
# ─────────────────────────────────────────────────────────────────────────────
 
def find_phrase(tokens: list[str], phrase: str) -> int:
    """
    Return the start index of the first occurrence of `phrase` (space-separated
    tokens) inside `tokens`. Returns -1 if not found.
    Uses exact token-level matching after lowercasing.
    """
    phrase_tokens = phrase.lower().split()
    n = len(phrase_tokens)
    for i in range(len(tokens) - n + 1):
        if tokens[i : i + n] == phrase_tokens:
            return i
    return -1

def find_any_phrase(tokens: list[str], phrases: list[str]) -> tuple[int, int]:
    """
    Greedy left-to-right search: find the FIRST occurrence of ANY phrase from
    the list (already sorted longest-first) inside `tokens`.
 
    Returns (start_index, length_in_tokens) or (-1, 0) if nothing found.
    """
    for phrase in phrases:
        idx = find_phrase(tokens, phrase)
        if idx != -1:
            return idx, len(phrase.split())
    return -1, 0
 
 
def find_all_occurrences(tokens: list[str], span: list[str]) -> list[int]:
    """
    Return all start indices where `span` (list of tokens) appears in `tokens`.
    """
    n = len(span)
    return [
        i for i in range(len(tokens) - n + 1)
        if tokens[i : i + n] == span
    ]

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# PER-TYPE LABELERS
# Each function receives the token list and returns a list[str] of BIO labels.
# If detection fails (edge case), returns all-O labels and logs a warning.
# ─────────────────────────────────────────────────────────────────────────────
 
def label_fluent(tokens: list[str], text: str) -> list[str]:
    """All tokens are fluent → all O."""
    return ["O"] * len(tokens)
 
 
def label_filled_pause(tokens: list[str], text: str) -> list[str]:
    """
    Standalone filled pauses (uh, um, hmm, eh, anu, ee …) that appear
    WITHOUT a preceding reparandum are labeled B-IM.
 
    A filled pause is 'standalone' when it is NOT immediately preceded by
    a content word that is itself followed by a different content word
    (which would make it an interregnum inside a repair — handled by
    label_substitution_repair instead).
 
    Simple rule: every token in FILLED_PAUSES → B-IM.
    """
    labels = ["O"] * len(tokens)
    for i, tok in enumerate(tokens):
        if tok in FILLED_PAUSES:
            labels[i] = "B-IM"
    return labels
 
 
def label_discourse_marker(tokens: list[str], text: str) -> list[str]:
    """
    Discourse markers (pokoknya, intinya, yaudah, kan, sih, deh …) that
    appear as sentence-level hedges (not inside a repair) are labeled B-IM
    on the first token and I-IM on any continuation tokens.
 
    Multi-token markers are handled by phrase-level matching. Adjacent
    independent markers (e.g. "aja deh") chain into one B-IM/I-IM span
    rather than emitting back-to-back B-IM tags.
    """
    labels = ["O"] * len(tokens)
    i = 0
    while i < len(tokens):
        matched = False
        for marker in DISCOURSE_MARKERS:
            mtkns = marker.split()
            mlen  = len(mtkns)
            if tokens[i : i + mlen] == mtkns:
                prev_is_im = i > 0 and labels[i - 1] in ("B-IM", "I-IM")
                labels[i] = "I-IM" if prev_is_im else "B-IM"
                for j in range(1, mlen):
                    labels[i + j] = "I-IM"   # continuation of a multi-token marker
                i += mlen
                matched = True
                break
        if not matched:
            i += 1
    return labels
 
 
def label_repetition(tokens: list[str], text: str) -> list[str]:
    """
    Detect exact repetitions of 1–3 adjacent tokens.
 
    Strategy:
      1. Scan with a sliding window of width w ∈ {3, 2, 1} (longest first).
      2. If tokens[i:i+w] == tokens[i+w:i+2w], the first occurrence is the
         reparandum → label B-RM + (w-1) × I-RM.
      3. Stop after the first match to avoid double-labeling.
 
    Edge case: repeated tokens separated by a filler (e.g. "saya, saya")
    are handled by checking with one gap token between the two spans.
    """
    labels = ["O"] * len(tokens)
    n = len(tokens)
 
    for w in [3, 2, 1]:
        found = False
        for i in range(n - 2 * w + 1):
            span_a = tokens[i     : i + w]
            span_b = tokens[i + w : i + 2 * w]
            if span_a == span_b:
                labels[i] = "B-RM"
                for j in range(1, w):
                    labels[i + j] = "I-RM"
                found = True
                break
        if found:
            break
 
    # Fallback: check for repetition with a filler in between
    # e.g. "nasinya, nasinya" after punctuation stripping → "nasinya nasinya"
    # already covered above. No extra logic needed.
 
    return labels
 
 
def label_substitution_repair(tokens: list[str], text: str) -> list[str]:
    """
    Substitution repair: [reparandum] [interregnum_bridge] [repair...]
 
    Strategy:
      1. Find the repair bridge (eh bukan / maksud saya / bukan / maaf …).
      2. Everything LEFT of the bridge (up to sentence start or a
         previous interregnum) is the reparandum → B-RM + I-RM.
      3. The bridge tokens → B-IM + I-IM.
      4. Everything RIGHT of the bridge → O (the repair, which is fluent).
 
    Handles multi-token bridges (e.g. "maksud saya" = 2 tokens).
    """
    labels = ["O"] * len(tokens)
 
    bridge_idx, bridge_len = find_any_phrase(tokens, REPAIR_BRIDGES)
 
    if bridge_idx == -1:
        # No bridge found — fall back to filled_pause labeling
        log.debug("substitution_repair: no bridge found in '%s', falling back", text)
        return label_filled_pause(tokens, text)
 
    # Label bridge as interregnum
    labels[bridge_idx] = "B-IM"
    for j in range(1, bridge_len):
        labels[bridge_idx + j] = "I-IM"
 
    # Label everything before the bridge as reparandum
    if bridge_idx > 0:
        labels[0] = "B-RM"
        for j in range(1, bridge_idx):
            labels[j] = "I-RM"
 
    # Everything after bridge stays O (repair is fluent)
    return labels
 
 
def label_restart(tokens: list[str], text: str) -> list[str]:
    """
    Restart: the speaker starts the utterance, abandons it mid-sentence,
    inserts a filler/bridge, then restarts with a fresh sentence.
 
    The abandoned start (reparandum) is at the utterance beginning.
    The filler (bridge) immediately follows the abandoned segment.
 
    Strategy:
      1. Find the first filled pause or repair bridge in the token sequence.
      2. Everything BEFORE the bridge (if any tokens exist there) = reparandum.
      3. The bridge = B-IM (+ I-IM for multi-token bridges).
      4. Everything after = O.
 
    This correctly handles:
      "saya mau, hmm, boleh saya pesan..."
       B-RM I-RM B-IM  O    O   O
    """
    labels = ["O"] * len(tokens)
 
    # Try filled pauses first (most common restart bridge in informal speech)
    bridge_idx = -1
    bridge_len = 0
 
    for i, tok in enumerate(tokens):
        if tok in FILLED_PAUSES:
            bridge_idx = i
            bridge_len = 1
            break
 
    # If no filled pause, try multi-word repair bridge phrases
    if bridge_idx == -1:
        bridge_idx, bridge_len = find_any_phrase(tokens, REPAIR_BRIDGES)
 
    if bridge_idx == -1 or bridge_idx == 0:
        # Bridge at position 0 means nothing precedes it — no reparandum
        # Label only the bridge itself as B-IM if it exists at position 0
        if bridge_idx == 0:
            labels[0] = "B-IM"
            for j in range(1, bridge_len):
                labels[j] = "I-IM"
        return labels
 
    # Reparandum = everything before the bridge
    labels[0] = "B-RM"
    for j in range(1, bridge_idx):
        labels[j] = "I-RM"
 
    # Bridge = B-IM (+ I-IM for multi-token bridges)
    labels[bridge_idx] = "B-IM"
    for j in range(1, bridge_len):
        labels[bridge_idx + j] = "I-IM"
 
    # Repair = O (already set)
    return labels
 
 
def label_insertion(tokens: list[str], text: str) -> list[str]:
    """
    Insertion repair: the speaker repeats a phrase but adds extra information
    in the repair.
 
    e.g. "minta mie rebus, hmm, mie rebus panas ya"
          B-RM  I-RM     B-IM  O    O    O    O
 
    Strategy:
      1. Find the filled pause / bridge in the middle.
      2. Collect 1–3 tokens before the bridge as reparandum.
      3. Label bridge as B-IM (+ I-IM for multi-token bridges).
      4. Everything after = O.
 
    Falls back to label_filled_pause if no bridge is found.
    """
    labels = ["O"] * len(tokens)
 
    # Find bridge
    bridge_idx = -1
    bridge_len = 0
    for i, tok in enumerate(tokens):
        if tok in FILLED_PAUSES:
            bridge_idx = i
            bridge_len = 1
            break
 
    if bridge_idx == -1:
        bridge_idx, bridge_len = find_any_phrase(tokens, REPAIR_BRIDGES)
 
    if bridge_idx == -1 or bridge_idx == 0:
        return label_filled_pause(tokens, text)
 
    # Reparandum = tokens immediately before the bridge (up to 3)
    rm_start = max(0, bridge_idx - 3)
    labels[rm_start] = "B-RM"
    for j in range(rm_start + 1, bridge_idx):
        labels[j] = "I-RM"
 
    # Bridge
    labels[bridge_idx] = "B-IM"
    for j in range(1, bridge_len):
        labels[bridge_idx + j] = "I-IM"
 
    return labels
 
 
def label_partial_word(tokens: list[str], text: str) -> list[str]:
    """
    Partial words end with a hyphen (na-, bi-, wou-).
    The truncated token = B-RM.
    The token immediately following (the complete repair word) = O.
    """
    labels = ["O"] * len(tokens)
    for i, tok in enumerate(tokens):
        if PARTIAL_WORD_RE.match(tok):          # ← uses dynamic constant
            labels[i] = "B-RM"
    return labels
 
 
def apply_filler_overlay(tokens: list[str], labels: list[str]) -> list[str]:
    """
    Post-pass applied AFTER the primary labeler, regardless of which
    disfluency_type was detected for the sentence.
 
    detect_disfluency_type picks exactly ONE type per sentence, so a
    sentence classified as e.g. "restart" never runs label_discourse_marker —
    any trailing hedge ("ya", "deh", "aja", "nih" …) on that sentence was
    left as O. This overlay scans only the tokens the primary labeler left
    as O and tags FILLED_PAUSES / DISCOURSE_MARKERS matches there, without
    touching any existing B-RM/I-RM/B-IM/I-IM span.
 
    Adjacent overlay-added matches (e.g. "aja deh") are merged into a single
    B-IM/I-IM span rather than emitting back-to-back B-IM tags, since BIO
    requires contiguous same-type tokens to chain as B- then I-.
    """
    labels = labels[:]
    n = len(tokens)
    i = 0
    while i < n:
        if labels[i] != "O":
            i += 1
            continue

        prev_is_im = i > 0 and labels[i - 1] in ("B-IM", "I-IM")

        if tokens[i] in FILLED_PAUSES:
            labels[i] = "I-IM" if prev_is_im else "B-IM"
            i += 1
            continue

        matched = False
        for marker in DISCOURSE_MARKERS:
            mtkns = marker.split()
            mlen  = len(mtkns)
            if tokens[i : i + mlen] == mtkns and all(
                labels[i + j] == "O" for j in range(mlen)
            ):
                labels[i] = "I-IM" if prev_is_im else "B-IM"
                for j in range(1, mlen):
                    labels[i + j] = "I-IM"
                i += mlen
                matched = True
                break
        if not matched:
            i += 1
    return labels

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# DISFLUENCY TYPE DETECTOR
# Since the 1k dataset has no disfluency_type field, we infer it from
# surface patterns. This mirrors the inverse of the generator logic.
# ─────────────────────────────────────────────────────────────────────────────
 
def detect_disfluency_type(tokens: list[str], text: str) -> str:
    """
    Heuristic classifier that maps a token sequence to one of the 8 types.
    Applied before labeling so the correct labeler is selected.
 
    Priority order matters:
      1. Partial word  — unambiguous (hyphen suffix)
      2. Repetition    — unambiguous (adjacent duplicate spans)
      3. Substitution  — repair bridge present + content before bridge
      4. Restart       — filled pause/bridge present + short prefix before it
      5. Insertion     — filled pause present + some context before it
      6. Filled pause  — any filler token present (no other signals)
      7. Discourse mrk — discourse marker token present (no filler/repair)
      8. Fluent        — none of the above
    """
    n = len(tokens)
 
    # 1. Partial word
    if any(PARTIAL_WORD_RE.match(t) for t in tokens):
        return "partial_word"
 
    # 2. Repetition — sliding window w ∈ {1,2,3}
    for w in [3, 2, 1]:
        for i in range(n - 2 * w + 1):
            if tokens[i : i + w] == tokens[i + w : i + 2 * w]:
                return "repetition"
 
    # Helper: does a repair bridge exist?
    bridge_idx, bridge_len = find_any_phrase(tokens, REPAIR_BRIDGES)
    has_bridge = bridge_idx > 0   # bridge must have content before it
 
    # Helper: does a filled pause exist?
    fp_positions = [i for i, t in enumerate(tokens) if t in FILLED_PAUSES]
    has_fp = len(fp_positions) > 0
 
    # 3. Substitution repair — bridge found AND content exists before it
    #    AND content exists after it (the actual repair)
    if has_bridge and bridge_idx + bridge_len < n:
        # Distinguish substitution vs restart:
        # restart: only 1–3 tokens before bridge (abandoned start)
        # substitution: meaningful content before bridge (reparandum)
        tokens_before_bridge = bridge_idx
        if tokens_before_bridge >= 2:
            return "substitution_repair"
        else:
            return "restart"
 
    # 4. Restart — filled pause present with 1–3 tokens before it
    if has_fp:
        first_fp = fp_positions[0]
        if 1 <= first_fp <= 4:
            return "restart"
 
    # 5. Insertion — filled pause present with more content before it
    if has_fp and fp_positions[0] > 1:
        return "insertion"
 
    # 6. Filled pause — filler present but no other structure
    if has_fp:
        return "filled_pause"
 
    # 7. Discourse marker
    for marker in DISCOURSE_MARKERS:
        mtkns = marker.split()
        if find_phrase(tokens, marker) != -1:
            return "discourse_marker"
 
    # 8. Fluent
    return "fluent"

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# DISPATCHER
# ─────────────────────────────────────────────────────────────────────────────
 
LABELERS = {
    "fluent"             : label_fluent,
    "filled_pause"       : label_filled_pause,
    "discourse_marker"   : label_discourse_marker,
    "repetition"         : label_repetition,
    "substitution_repair": label_substitution_repair,
    "restart"            : label_restart,
    "insertion"          : label_insertion,
    "partial_word"       : label_partial_word,
}
 
 
def label_record(record: dict) -> dict:
    """
    Full pipeline for a single record:
      1. Normalize + tokenize text
      2. Detect disfluency type from surface patterns
      3. Apply the corresponding labeler
      4. Overlay filler/discourse-marker tags onto any tokens the primary
         labeler left as O (detect_disfluency_type only picks one type per
         sentence, so e.g. a "restart" sentence's trailing "ya"/"deh" would
         otherwise never get tagged)
      5. Validate BIO constraints
      6. Return enriched record
    """
    tokens = normalize_and_tokenize(record["text_normalized"])
 
    if not tokens:
        log.warning("Empty token sequence for id=%s", record["id"])
        return {**record, "tokens": [], "labels": [], "disfluency_type": "fluent"}
 
    disfl_type = detect_disfluency_type(tokens, record["text_normalized"])
    labeler    = LABELERS[disfl_type]
    labels     = labeler(tokens, record["text_normalized"])
 
    # Safety net: length mismatch should never happen, but guard anyway
    if len(labels) != len(tokens):
        log.error(
            "id=%s: label/token length mismatch (%d vs %d). Defaulting to all-O.",
            record["id"], len(labels), len(tokens)
        )
        labels = ["O"] * len(tokens)
 
    labels = apply_filler_overlay(tokens, labels)
 
    return {
        "id"             : record["id"],
        "text"           : record["text_normalized"],
        "style"          : record["style"],
        "domain"         : record["domain"],
        "tokens"         : tokens,
        "labels"         : labels,
        "disfluency_type": disfl_type,
    }

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# BIO VALIDATION
# ─────────────────────────────────────────────────────────────────────────────
 
def validate_bio(tokens: list[str], labels: list[str], record_id: int) -> list[str]:
    """
    Enforce BIO structural constraints and auto-correct violations:
 
    Rule 1: I-RM must be preceded by B-RM or I-RM; I-IM must be preceded
            by B-IM or I-IM.
            Violation → convert orphan I-X to B-X.
 
    Rule 2: Labels list must be same length as tokens list.
            Already checked in label_record(); included here for safety.
 
    Rule 3: No unknown labels.
            Unknown labels → O.
 
    Returns the (possibly corrected) labels list.
    """
    valid_labels = {"O", "B-RM", "I-RM", "B-IM", "I-IM"}
    corrected = labels[:]
 
    for i, lbl in enumerate(corrected):
        # Rule 3: unknown label
        if lbl not in valid_labels:
            log.warning("id=%s pos=%d: unknown label '%s' → O", record_id, i, lbl)
            corrected[i] = "O"
            continue
 
        # Rule 1: orphan I-X
        if lbl == "I-RM":
            if i == 0 or corrected[i - 1] not in ("B-RM", "I-RM"):
                log.debug("id=%s pos=%d: orphan I-RM → B-RM", record_id, i)
                corrected[i] = "B-RM"
        elif lbl == "I-IM":
            if i == 0 or corrected[i - 1] not in ("B-IM", "I-IM"):
                log.debug("id=%s pos=%d: orphan I-IM → B-IM", record_id, i)
                corrected[i] = "B-IM"
 
    return corrected

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# SPOT-CHECK PRINTER
# ─────────────────────────────────────────────────────────────────────────────
 
def print_spot_check(records: list[dict], n_per_type: int = 3) -> None:
    """
    Print n_per_type annotated examples per disfluency type so you can
    visually verify label correctness before committing to training.
    """
    from collections import defaultdict
    by_type = defaultdict(list)
    for r in records:
        by_type[r["disfluency_type"]].append(r)
 
    print("\n" + "=" * 70)
    print("SPOT-CHECK: sample annotations per disfluency type")
    print("=" * 70)
 
    type_order = [
        "fluent", "filled_pause", "discourse_marker", "repetition",
        "substitution_repair", "restart", "insertion", "partial_word",
    ]
 
    for dtype in type_order:
        samples = by_type.get(dtype, [])
        print(f"\n── {dtype.upper()} ({len(samples)} records total) ──")
        for rec in samples[:n_per_type]:
            pairs = list(zip(rec["tokens"], rec["labels"]))
            # Color-code in terminal: red = disfluent, normal = fluent
            display = []
            for tok, lbl in pairs:
                if lbl != "O":
                    display.append(f"[{tok}|{lbl}]")
                else:
                    display.append(tok)
            print(f"  id={rec['id']:4d}  {' '.join(display)}")

In [9]:
from collections import Counter

# ─────────────────────────────────────────────────────────────────────────────
# STATISTICS
# ─────────────────────────────────────────────────────────────────────────────
 
def print_statistics(records: list[dict]) -> None:
    """Print dataset statistics after labeling."""
    total_tokens = sum(len(r["tokens"]) for r in records)
    label_counts = Counter(lbl for r in records for lbl in r["labels"])
    type_counts  = Counter(r["disfluency_type"] for r in records)
    style_counts = Counter(r["style"] for r in records)
 
    print("\n" + "=" * 70)
    print("DATASET STATISTICS")
    print("=" * 70)
    print(f"  Total records       : {len(records)}")
    print(f"  Total tokens        : {total_tokens}")
    print(f"  Avg tokens/record   : {total_tokens / len(records):.1f}")
    print()
    print("  Style distribution:")
    for k, v in sorted(style_counts.items()):
        print(f"    {k:12s}: {v:5d}  ({v/len(records)*100:.1f}%)")
    print()
    print("  Disfluency type distribution:")
    for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"    {k:25s}: {v:5d}  ({v/len(records)*100:.1f}%)")
    print()
    print("  Label distribution (token-level):")
    for k, v in sorted(label_counts.items()):
        print(f"    {k:8s}: {v:6d}  ({v/total_tokens*100:.1f}%)")
 
    # Check for records with no disfluency labels that were expected to have them
    non_fluent_all_O = [
        r for r in records
        if r["disfluency_type"] != "fluent" and all(l == "O" for l in r["labels"])
    ]
    if non_fluent_all_O:
        print(f"\n  ⚠️  {len(non_fluent_all_O)} non-fluent records got all-O labels "
              f"(detection fallback). Sample ids: "
              f"{[r['id'] for r in non_fluent_all_O[:5]]}")
    else:
        print("\n  ✅  All non-fluent records have at least one disfluency label.")

In [10]:
from pathlib import Path

input_path  = Path("../data/synthetic_id_formal_informal_normalized.jsonl")
output_path = Path("../data/disfluency_dataset_bio.jsonl")

raw_records = []
with open(input_path, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))
raw_records[0]

{'id': 1,
 'text': 'Tolong pesankan nasi kuning tiga porsi, untuk dibawa pulang ya.',
 'style': 'formal',
 'domain': 'restoran',
 'text_normalized': 'tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya'}

In [11]:
labeled = []
for rec in raw_records:
    result = label_record(rec)
    result["labels"] = validate_bio(
        result["tokens"], result["labels"], result["id"]
    )
    labeled.append(result)
labeled[0]

{'id': 1,
 'text': 'tolong pesankan nasi kuning tiga porsi untuk dibawa pulang ya',
 'style': 'formal',
 'domain': 'restoran',
 'tokens': ['tolong',
  'pesankan',
  'nasi',
  'kuning',
  'tiga',
  'porsi',
  'untuk',
  'dibawa',
  'pulang',
  'ya'],
 'labels': ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-IM'],
 'disfluency_type': 'discourse_marker'}

In [12]:
with open(output_path, "w", encoding="utf-8") as f:
    for rec in labeled:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

In [13]:
print_statistics(labeled)


DATASET STATISTICS
  Total records       : 1100
  Total tokens        : 10636
  Avg tokens/record   : 9.7

  Style distribution:
    formal      :   550  (50.0%)
    informal    :   550  (50.0%)

  Disfluency type distribution:
    discourse_marker         :   350  (31.8%)
    restart                  :   250  (22.7%)
    fluent                   :   161  (14.6%)
    repetition               :   131  (11.9%)
    substitution_repair      :   114  (10.4%)
    filled_pause             :    57  (5.2%)
    insertion                :    37  (3.4%)

  Label distribution (token-level):
    B-IM    :   1247  (11.7%)
    B-RM    :    523  (4.9%)
    I-IM    :     74  (0.7%)
    I-RM    :    978  (9.2%)
    O       :   7814  (73.5%)

  ✅  All non-fluent records have at least one disfluency label.


In [14]:
print_spot_check(labeled, n_per_type=3)


SPOT-CHECK: sample annotations per disfluency type

── FLUENT (161 records total) ──
  id=  14  apakah ada promo hari ini kalau ada saya mau pesan smoothie
  id=  24  intinya saya mau nasi kuning minumannya bebas saja
  id=  28  pak ada cumi goreng gak kalo ada gue mau satu

── FILLED_PAUSE (57 records total) ──
  id=  53  [eh|B-IM] bentar [hmm|B-IM] gue mau liat menu dulu [ya|B-IM] bang
  id=  66  [eh|B-IM] cancel yang tadi gue ganti jadi ayam penyet [eh|B-IM] nasi uduk [deh|B-IM]
  id=  74  [eh|B-IM] mbak ganti [ya|B-IM] dari ikan goreng ke tumis kangkung

── DISCOURSE_MARKER (350 records total) ──
  id=   1  tolong pesankan nasi kuning tiga porsi untuk dibawa pulang [ya|B-IM]
  id=   3  pesenin ikan goreng empat buat meja gue [ya|B-IM] pak
  id=   5  kak nasinya dipisah [ya|B-IM] dari nasi goreng pete nya

── REPETITION (131 records total) ──
  id=   2  [saya|B-RM] [mau|I-RM] saya mau memesan pasta dua mangkok
  id=  18  saya dan teman saya ingin memesan ayam [rica|B-RM] rica masin

## Step 2 — Tokenizer alignment

IndoBERT's WordPiece tokenizer splits a word into multiple subword pieces (e.g. `"pesankan"` -> `["pes", "##an", "##kan"]`). Our BIO labels are per-word, so each subword piece needs the label propagated from its parent word. We follow the standard NER convention: the **first** subword of a word keeps the word's label id, every **continuation** subword gets `-100` so the loss function ignores it. `[CLS]`/`[SEP]`/padding also get `-100`.

In [15]:
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 64

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

LABEL_LIST = ["O", "B-RM", "I-RM", "B-IM", "I-IM"]
label2id = {lbl: i for i, lbl in enumerate(LABEL_LIST)}
id2label = {i: lbl for lbl, i in label2id.items()}
label2id

{'O': 0, 'B-RM': 1, 'I-RM': 2, 'B-IM': 3, 'I-IM': 4}

In [16]:
def align_labels_with_tokens(tokens: list[str], labels: list[str]) -> dict:
    """
    Tokenize a pre-split word list and align word-level BIO labels to
    subword tokens using word_ids().

    First subword of each word -> label2id[word_label]
    Continuation subwords      -> -100 (ignored by CrossEntropyLoss)
    Special tokens ([CLS]/[SEP]/[PAD]) -> -100
    """
    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

    word_ids = encoding.word_ids()
    aligned_labels = []
    prev_word_id = None

    for word_id in word_ids:
        if word_id is None:
            aligned_labels.append(-100)
        elif word_id != prev_word_id:
            aligned_labels.append(label2id[labels[word_id]])
        else:
            aligned_labels.append(-100)
        prev_word_id = word_id

    encoding["labels"] = aligned_labels
    return encoding

### Sanity check on one record

Verify subword pieces line up with the right propagated label and that continuation pieces are masked with `-100`.

In [17]:
sample = labeled[3]  # id=4, "gue mau pesen hmm thai tea dua gelas ya bu" — multiple disfluency tags
enc = align_labels_with_tokens(sample["tokens"], sample["labels"])

subword_tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
for tok, lbl_id in zip(subword_tokens, enc["labels"]):
    if tok == tokenizer.pad_token:
        break
    lbl = id2label[lbl_id] if lbl_id != -100 else "-100"
    print(f"  {tok:15s} {lbl}")

  [CLS]           -100
  gue             B-RM
  mau             I-RM
  pesen           I-RM
  hmm             B-IM
  thai            O
  tea             O
  dua             O
  gelas           O
  ya              B-IM
  bu              O
  [SEP]           -100


### Align the full dataset and inspect subword growth

Run alignment over all 1100 records, then check how many tokens grew into multiple subwords (the actual problem Step 2 solves) and confirm `len(input_ids) == len(labels)` holds everywhere.

In [18]:
aligned_records = []
for rec in labeled:
    enc = align_labels_with_tokens(rec["tokens"], rec["labels"])
    aligned_records.append({
        "id": rec["id"],
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": enc["labels"],
    })

# Structural guarantee: every record's input_ids/labels are the same length
assert all(len(r["input_ids"]) == len(r["labels"]) for r in aligned_records)

n_subword_growth = sum(
    1 for rec, aligned in zip(labeled, aligned_records)
    if sum(1 for w in aligned["labels"] if w != -100) != len(rec["tokens"])
    # mismatch would mean a word's first-subword marker was lost — should never trigger
)
n_truncated = sum(
    1 for rec in aligned_records
    if rec["attention_mask"][-1] == 1 and rec["input_ids"][-1] != tokenizer.sep_token_id
)

print(f"Aligned records       : {len(aligned_records)}")
print(f"Length mismatches     : {n_subword_growth}  (should be 0)")
print(f"Truncated at max_len  : {n_truncated}")

Aligned records       : 1100
Length mismatches     : 0  (should be 0)
Truncated at max_len  : 0


### Save aligned dataset

Output feeds directly into Step 3 (dataset splitting) — already tokenized and padded to `MAX_LENGTH`, ready for a `datasets.Dataset` / DataLoader.

In [19]:
aligned_output_path = Path("../data/disfluency_dataset_aligned.jsonl")

with open(aligned_output_path, "w", encoding="utf-8") as f:
    for rec in aligned_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(aligned_records)} aligned records -> {aligned_output_path}")

Saved 1100 aligned records -> ../data/disfluency_dataset_aligned.jsonl


## Step 3 — Dataset splitting

Split the 1100 aligned records into train/val/test (80/10/10), stratified by `disfluency_type` so rare classes (`insertion`=37, `filled_pause`=57) keep proportional representation in every split instead of landing entirely in one partition.

In [20]:
from sklearn.model_selection import train_test_split

SEED = 42

# disfluency_type lives on `labeled`, not on `aligned_records` — join by id
type_by_id = {rec["id"]: rec["disfluency_type"] for rec in labeled}
strat_labels = [type_by_id[rec["id"]] for rec in aligned_records]

# 80/10/10: first peel off 20% (val+test), then split that 20% in half
train_records, valtest_records, train_strat, valtest_strat = train_test_split(
    aligned_records,
    strat_labels,
    test_size=0.2,
    stratify=strat_labels,
    random_state=SEED,
)
val_records, test_records = train_test_split(
    valtest_records,
    test_size=0.5,
    stratify=valtest_strat,
    random_state=SEED,
)

print(f"train: {len(train_records)}  val: {len(val_records)}  test: {len(test_records)}")

train: 880  val: 110  test: 110


### Verify stratification held

Compare `disfluency_type` proportions across splits — should be close to identical, including for the rarest classes.

In [21]:
def type_proportions(records: list[dict]) -> dict:
    ids = {rec["id"] for rec in records}
    types = [type_by_id[i] for i in ids]
    c = Counter(types)
    n = len(types)
    return {k: round(v / n * 100, 1) for k, v in sorted(c.items(), key=lambda x: -x[1])}

print("train:", type_proportions(train_records))
print("val  :", type_proportions(val_records))
print("test :", type_proportions(test_records))

train: {'discourse_marker': 31.8, 'restart': 22.7, 'fluent': 14.7, 'repetition': 11.9, 'substitution_repair': 10.3, 'filled_pause': 5.1, 'insertion': 3.4}
val  : {'discourse_marker': 31.8, 'restart': 22.7, 'fluent': 14.5, 'repetition': 11.8, 'substitution_repair': 10.9, 'filled_pause': 5.5, 'insertion': 2.7}
test : {'discourse_marker': 31.8, 'restart': 22.7, 'fluent': 14.5, 'repetition': 11.8, 'substitution_repair': 10.0, 'filled_pause': 5.5, 'insertion': 3.6}


### Save splits

Each file holds tokenized + aligned records (`input_ids`, `attention_mask`, `labels`) ready for `datasets.Dataset.from_list()` in Step 4/5.

In [22]:
splits = {
    "train": train_records,
    "val": val_records,
    "test": test_records,
}

for name, records in splits.items():
    out_path = Path(f"../data/disfluency_dataset_{name}.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Saved {len(records):4d} records -> {out_path}")

Saved  880 records -> ../data/disfluency_dataset_train.jsonl
Saved  110 records -> ../data/disfluency_dataset_val.jsonl
Saved  110 records -> ../data/disfluency_dataset_test.jsonl


## Step 4 — Class imbalance

Token-level label distribution is heavily skewed toward `O` (~74%), and `I-IM` is rare (<1%, only 51 tokens in train). Without correction the model can minimize loss by collapsing toward predicting `O` everywhere and never learning the rare classes. We compute inverse-frequency class weights from the **train split only** (never val/test, to avoid leaking split-specific statistics into training) and pass them to `CrossEntropyLoss`.

In [23]:
import torch
from collections import Counter

def compute_label_counts(jsonl_path: str) -> Counter:
    """Count label ids across a split, excluding -100 (ignored/padding)."""
    counts = Counter()
    with open(jsonl_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            for lbl_id in rec["labels"]:
                if lbl_id != -100:
                    counts[lbl_id] += 1
    return counts


train_label_counts = compute_label_counts("../data/disfluency_dataset_train.jsonl")
for lbl_id in range(len(LABEL_LIST)):
    print(f"  {id2label[lbl_id]:6s}: {train_label_counts[lbl_id]:6d}")

  O     :   6268
  B-RM  :    419
  I-RM  :    792
  B-IM  :    991
  I-IM  :     51


In [24]:
# Inverse-frequency weighting: weight_c = total / (n_classes * count_c)
# Common normalization so weights average to ~1 instead of blowing up the loss scale.
total_tokens = sum(train_label_counts.values())
n_classes = len(LABEL_LIST)

class_weights = torch.tensor(
    [total_tokens / (n_classes * train_label_counts[i]) for i in range(n_classes)],
    dtype=torch.float,
)

for lbl_id, weight in enumerate(class_weights.tolist()):
    print(f"  {id2label[lbl_id]:6s}: weight={weight:.3f}")

loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

  O     : weight=0.272
  B-RM  : weight=4.067
  I-RM  : weight=2.152
  B-IM  : weight=1.720
  I-IM  : weight=33.416


### Sanity check

Confirm the rarest class (`I-IM`) gets the largest weight, the majority class (`O`) gets the smallest, and `ignore_index=-100` matches the padding/continuation convention from Step 2.

In [25]:
assert class_weights.argmax().item() == label2id["I-IM"], "I-IM should get the highest weight"
assert class_weights.argmin().item() == label2id["O"], "O should get the lowest weight"
assert loss_fn.ignore_index == -100

print("class_weights tensor:", class_weights)
print("Step 4 sanity checks passed.")

class_weights tensor: tensor([ 0.2719,  4.0673,  2.1518,  1.7197, 33.4157])
Step 4 sanity checks passed.


## Step 5 — Model architecture

Load `indobenchmark/indobert-base-p1` via `AutoModelForTokenClassification` — BERT backbone + a linear classification head (with dropout) on top of every token's hidden state, predicting one of the 5 BIO label ids. We full fine-tune (no frozen layers): the dataset is small (1100 records) but full fine-tuning is still the standard baseline for this dataset size, and the class weights from Step 4 cover the imbalance risk.

In [26]:
from transformers import AutoModelForTokenClassification

device = (
    torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cuda") if torch.cuda.is_available()
    else torch.device("cpu")
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device          : {device}")
print(f"total params    : {n_params:,}")
print(f"trainable params: {n_trainable:,}  (full fine-tune — should equal total)")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


device          : mps
total params    : 123,854,597
trainable params: 123,854,597  (full fine-tune — should equal total)


### Sanity check: forward pass on a real batch

Confirm logits come out shaped `(batch, seq_len, num_labels)` and that the weighted loss from Step 4 computes against the model's own logits without shape errors — this is the exact call shape Step 6 (training) will make every step.

In [27]:
sample_batch = aligned_records[:4]
batch_input_ids = torch.tensor([r["input_ids"] for r in sample_batch], device=device)
batch_attention_mask = torch.tensor([r["attention_mask"] for r in sample_batch], device=device)
batch_labels = torch.tensor([r["labels"] for r in sample_batch], device=device)

model.eval()
with torch.no_grad():
    outputs = model(input_ids=batch_input_ids, attention_mask=batch_attention_mask)

logits = outputs.logits
print(f"logits shape: {tuple(logits.shape)}  (expect (4, {MAX_LENGTH}, {len(LABEL_LIST)}))")
assert logits.shape == (4, MAX_LENGTH, len(LABEL_LIST))

# Confirm Step 4's weighted loss_fn accepts these logits directly:
# CrossEntropyLoss expects (N, C, ...) vs target (N, ...), so flatten batch+seq dims.
loss_fn = loss_fn.to(device)
sample_loss = loss_fn(logits.view(-1, len(LABEL_LIST)), batch_labels.view(-1))
print(f"sample weighted loss: {sample_loss.item():.4f}")

model.train()
print("Step 5 sanity checks passed.")

logits shape: (4, 64, 5)  (expect (4, 64, 5))
sample weighted loss: 1.9655
Step 5 sanity checks passed.
